In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# 03 · E002: Does slicing the image help find far-away people?

**The question:** far-away people are only a few pixels wide, and YOLO misses most of them (E001: only 36–42% of far-half people found). A common fix is to cut the picture into small overlapping tiles and show each tile to YOLO enlarged (this idea is called SAHI). Does it find more people?

**The catch we noticed:** our frames are already shown to YOLO at their real size. Cutting into tiles only helps if each tile is also **enlarged**. So we test four settings: no tiles (the baseline), tiles at real size (should change nothing: a control), tiles enlarged 2×, and smaller tiles enlarged 4×.

**Rule decided in advance:** if the share of people found rises by more than 5 percentage points, slicing joins our baseline. If not, slicing does not fix the problem and we need density estimation. Both outcomes count as results. We also record the time per frame, because slicing costs computing time.

## Step 1: Get our code from GitHub

**In simple words:** Kaggle starts empty every session. This downloads our repository (our own code and the annotation file). Run it first, and again after every restart.

In [2]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/SagarInspires/crowd-intelligence.git"
REPO_DIR = "/kaggle/working/crowd-intelligence"

def sync_repo():
    """Clone the repo on a fresh session, or pull the newest code if it is already
    there, then make the `src` folder importable."""
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

sync_repo()
print("repo ready at", REPO_DIR)
print("tiling.py present:", os.path.exists(f"{REPO_DIR}/src/models/tiling.py"))   # must be True: this is the new tiling code

Cloning into '/kaggle/working/crowd-intelligence'...


repo ready at /kaggle/working/crowd-intelligence
tiling.py present: True


## Step 2: Install the YOLO library

**In simple words:** YOLO comes from a library called `ultralytics`, which Kaggle does not include. This installs it once per session. It needs **Internet ON** in the notebook settings (right sidebar). If you restart the notebook, run this cell again.

In [3]:
# Install the YOLO library (one time per session; ignore dependency-conflict warnings)
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 4.8 MB/s eta 0:00:00


## Step 3: Load ground truth and helper functions

**In simple words:** we read the heads we clicked using VIA annotator by Oxford, in 15 frames and the ignore areas from GitHub, and load the scoring code we wrote in E001 plus the new tiling code. We also define the same 5 tuning frames and 10 reporting frames as before, so the results are directly comparable with E001. The printout should say 6816 points, 15 frames.

In [4]:
import time
from pathlib import Path                                       # explicit, so this cell does not depend on Cell 2's imports
import numpy as np, pandas as pd, cv2, torch
from ultralytics import YOLO
from src.data.via import load_ground_truth
from src.evaluation.detection import eval_frame, summarise, cluster_bootstrap_ci
from src.models.tiling import detect_tiled

DATA = Path(f"{REPO_DIR}/data/your_video")
pts, ign, _ = load_ground_truth(DATA)                          # clicked heads and ignore polygons
SHAPE = cv2.imread(str(DATA / "annotated_frames/frame_00000.jpg")).shape[:2]   # (height, width)
OUT = Path("/kaggle/working/outputs/sahi"); OUT.mkdir(parents=True, exist_ok=True)
ids = sorted(int(i) for i in pts.frame_id.unique())
tune_ids = ids[1::3]                                           # same 5 tuning frames as E001
test_ids = [i for i in ids if i not in tune_ids]               # same 10 reporting frames as E001
far_y = SHAPE[0] / 2                                           # rows above the middle count as "far"
thresholds = [0.001, 0.002, 0.005, 0.01, 0.02, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
print(len(pts), "points,", len(ids), "frames, size (H, W) =", SHAPE, "| GPU:", torch.cuda.is_available())

def rows_for(det, fids, thr):
    """One metrics row per frame at confidence cutoff thr (ignore areas excluded)."""
    return pd.DataFrame([dict(frame_id=i, **eval_frame(
        pts[pts.frame_id == i][["x", "y"]].to_numpy(float), *det[f"frame_{i:05d}"],
        ign[ign.frame_id == i]["vertices"].tolist(), thr, far_y)) for i in fids])

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
6816 points, 15 frames, size (H, W) = (1920, 1080) | GPU: True


## Step 4: Run the four settings on the 15 frames

**In simple words:** for both detectors we run each setting on every frame and keep almost every candidate box (confidence 0.001 and above); the real cutoff is chosen later on the tuning frames. We time each setting. The tile settings cut the frame into overlapping tiles, show each tile to YOLO at the chosen size, then put the boxes back in place. Each person is kept by exactly one tile, so no person is counted twice and no two neighbours are merged.

**Read the timing:** the 4× setting costs the most; if it does not help much it is not worth it. The printed versions (ultralytics, torch, GPU) go in the experiment log, because detector results depend on them.

In [5]:
import ultralytics

MODELS = ["yolov8x.pt", "yolo26x.pt"]
CONFIGS = {                                                    # name: (tile size or None for whole frame, detector input size)
    "full frame @1920 (E001 baseline)": (None, 1920),
    "tiles 640 @640 (1x, control)":     (640, 640),
    "tiles 640 @1280 (2x)":             (640, 1280),
    "tiles 320 @1280 (4x)":             (320, 1280),
}
MAX_DET = 3000
frames = {i: cv2.imread(str(DATA / f"annotated_frames/frame_{i:05d}.jpg")) for i in ids}   # BGR images in memory
print(f"ultralytics {ultralytics.__version__} | torch {torch.__version__} | "
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")   # record for the write-up

def detect_config(model, tile, imgsz):
    """Detect people in all 15 frames with one setting. Returns {frame key: (boxes xyxy, scores)}."""
    out = {}
    for i, img in frames.items():
        if tile is None:                                       # whole frame at the given input size
            r = model.predict(img, imgsz=imgsz, classes=[0], conf=0.001, max_det=MAX_DET, verbose=False)[0]
            out[f"frame_{i:05d}"] = (r.boxes.xyxy.cpu().numpy().astype(np.float32), r.boxes.conf.cpu().numpy().astype(np.float32))
        else:                                                  # sliced detection
            out[f"frame_{i:05d}"] = detect_tiled(model, img, tile, imgsz, overlap=0.2, conf_floor=0.001, max_det=MAX_DET)
    return out

def warm_up(model):
    """Run a few throwaway predictions so CUDA start-up and kernel compilation are not counted in any setting's timing."""
    first = next(iter(frames.values()))
    for size in (1920, 1280, 640):
        model.predict(first, imgsz=size, classes=[0], conf=0.25, verbose=False)

def sync():
    """Wait for all queued GPU work to finish, so a timer stops when the work is really done."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()

dets, secs = {}, {}
for name in MODELS:
    model = YOLO(name)
    warm_up(model)                                             # fair timing: every setting starts from a warm GPU
    for cname, (tile, imgsz) in CONFIGS.items():
        sync(); t0 = time.perf_counter()
        dets[(name, cname)] = detect_config(model, tile, imgsz)
        sync(); secs[(name, cname)] = (time.perf_counter() - t0) / len(frames)
        n = [len(v[0]) for v in dets[(name, cname)].values()]
        flag = "   <-- REACHED max_det, recall is capped!" if max(n) >= MAX_DET else ""
        print(f"{name:11s} {cname:34s} boxes/frame {min(n)}-{max(n)}   {secs[(name, cname)]:.2f} s/frame{flag}")
    del model; torch.cuda.empty_cache()                        # free GPU memory before loading the next detector

ultralytics 8.4.165 | torch 2.10.0+cu128 | GPU: Tesla T4
yolov8x.pt  full frame @1920 (E001 baseline)   boxes/frame 979-1245   0.31 s/frame
yolov8x.pt  tiles 640 @640 (1x, control)       boxes/frame 1048-1242   0.52 s/frame
yolov8x.pt  tiles 640 @1280 (2x)               boxes/frame 2049-2364   2.00 s/frame
yolov8x.pt  tiles 320 @1280 (4x)               boxes/frame 7190-8636   8.72 s/frame   <-- REACHED max_det, recall is capped!
yolo26x.pt  full frame @1920 (E001 baseline)   boxes/frame 1544-1802   0.36 s/frame
yolo26x.pt  tiles 640 @640 (1x, control)       boxes/frame 944-1229   0.55 s/frame
yolo26x.pt  tiles 640 @1280 (2x)               boxes/frame 2490-3441   2.17 s/frame   <-- REACHED max_det, recall is capped!
yolo26x.pt  tiles 320 @1280 (4x)               boxes/frame 8687-9807   8.69 s/frame   <-- REACHED max_det, recall is capped!


## Step 4b: Check that no detector call was cut off

**In simple words:** the warning printed above compared a whole frame's boxes with a limit that applies to each **tile**, so it was a false alarm. This cell checks properly: it runs the most crowded frame (frame 0) with the most extreme setting (4× tiles) and records the largest number of boxes any single tile returned. If that number is well below 3000, nothing was cut off.

In [6]:
class CapProbe:
    """Wraps a detector and remembers the largest number of boxes any single predict() call returned."""
    def __init__(self, model):
        self.model, self.max_seen = model, 0
    def predict(self, *args, **kwargs):
        res = self.model.predict(*args, **kwargs)
        self.max_seen = max(self.max_seen, len(res[0].boxes))     # boxes returned by this one call (one tile or one frame)
        return res

for name in MODELS:
    probe = CapProbe(YOLO(name))
    for cname in ("full frame @1920 (E001 baseline)", "tiles 640 @1280 (2x)", "tiles 320 @1280 (4x)"):
        tile, imgsz = CONFIGS[cname]
        if tile is None:
            probe.predict(frames[0], imgsz=imgsz, classes=[0], conf=0.001, max_det=MAX_DET, verbose=False)
        else:
            detect_tiled(probe, frames[0], tile, imgsz, overlap=0.2, conf_floor=0.001, max_det=MAX_DET)
        print(f"{name:11s} {cname:34s} largest single call: {probe.max_seen:5d} boxes (limit {MAX_DET})"
              + ("   <-- CAPPED" if probe.max_seen >= MAX_DET else ""))
        probe.max_seen = 0

yolov8x.pt  full frame @1920 (E001 baseline)   largest single call:  1245 boxes (limit 3000)
yolov8x.pt  tiles 640 @1280 (2x)               largest single call:   924 boxes (limit 3000)
yolov8x.pt  tiles 320 @1280 (4x)               largest single call:  1432 boxes (limit 3000)
yolo26x.pt  full frame @1920 (E001 baseline)   largest single call:  1658 boxes (limit 3000)
yolo26x.pt  tiles 640 @1280 (2x)               largest single call:  1762 boxes (limit 3000)
yolo26x.pt  tiles 320 @1280 (4x)               largest single call:  2164 boxes (limit 3000)


## Step 5: Score every setting the same way as E001

**In simple words:** for each detector and each setting we pick the confidence cutoff on the 5 tuning frames, then report on the other 10 frames only. Same rules as E001: ignore areas excluded, one clicked head can match only one box.

**How to read the table:**
- `recall` and `recall_far`: the share of clicked people found (overall and far half). **This is the number our rule is about.**
- `d_recall_pp`, `d_far_pp`: the change compared with the no-tiles baseline, in percentage points.
- `MAE`, `bias`: counting error. `precision` is a lower bound (we did not click every person).
- `s_per_frame`: computing cost of our simple one-tile-at-a-time code (a batched version would be faster, so treat it as an upper bound).
- **Rule:** slicing joins the baseline if `d_recall_pp` is above +5 for a setting. A cutoff of 0.001 (the lowest we tried) would mean the search grid is too narrow, so tell me.

In [7]:
def best_cutoff(det):
    """Pick the cutoff with the best F1 on the tuning frames only."""
    return max(thresholds, key=lambda t: summarise(rows_for(det, tune_ids, t))["F1"])

rows = []
for (name, cname), det in dets.items():
    t = best_cutoff(det)
    res = rows_for(det, test_ids, t); S = summarise(res)
    lo, hi = cluster_bootstrap_ci((res.n_det - res.n_true).abs())
    rows.append(dict(model=name, setting=cname, cutoff=t, MAE=S["MAE"], MAE_CI=f"{lo:.0f}-{hi:.0f}", bias=S["bias"],
                     precision=S["precision"], recall=100 * S["recall"], recall_far=100 * S["recall_far"],
                     recall_near=100 * S["recall_near"], s_per_frame=secs[(name, cname)]))
table = pd.DataFrame(rows)
base = table[table.setting.str.startswith("full frame")].set_index("model")
table["d_recall_pp"] = table.apply(lambda r: r.recall - base.loc[r.model, "recall"], axis=1)      # change vs no-tiles baseline
table["d_far_pp"] = table.apply(lambda r: r.recall_far - base.loc[r.model, "recall_far"], axis=1)
table.to_csv(OUT / "E002_summary.csv", index=False)
print("tuning frames:", tune_ids, "| reporting frames:", test_ids)
print(table.round(2).to_string(index=False))

tuning frames: [93, 372, 651, 930, 1209] | reporting frames: [0, 186, 279, 465, 558, 744, 837, 1023, 1116, 1302]
     model                          setting  cutoff  MAE MAE_CI  bias  precision  recall  recall_far  recall_near  s_per_frame  d_recall_pp  d_far_pp
yolov8x.pt full frame @1920 (E001 baseline)    0.00 18.7  11-28 -15.1       0.68   65.80       36.16        78.72         0.31         0.00      0.00
yolov8x.pt     tiles 640 @640 (1x, control)    0.00 33.7  22-47  33.7       0.63   67.39       35.15        81.45         0.52         1.59     -1.01
yolov8x.pt             tiles 640 @1280 (2x)    0.02 33.5  19-49 -32.9       0.67   62.64       33.64        75.27         2.00        -3.16     -2.52
yolov8x.pt             tiles 320 @1280 (4x)    0.10 35.1  26-44  12.9       0.64   65.45       36.52        78.06         8.72        -0.35      0.36
yolo26x.pt full frame @1920 (E001 baseline)    0.02 31.3  15-48 -29.5       0.76   71.08       47.38        81.42         0.36         0.

## Step 6: Is the improvement real, or luck of 10 frames?

**In simple words:** with only 10 test frames, a gain of a few points might come from which frames we happened to pick. So we redo the comparison many times, each time re-drawing 10 frames at random (with repeats) from our 10, and record the gain each time. The range that covers 95% of those gains is the **confidence range**. If the whole range is above zero, the gain is reliable for this video. If the range includes zero, we cannot say the improvement is real. We compare each tiled setting with the full-frame baseline of the same detector, using the same cutoffs as in the table.

In [8]:
# ===== Cell 7: paired statistics for the tiling experiment (E002) =====
import itertools
import numpy as np
import pandas as pd

def paired_delta(A, B, n_boot=10_000, seed=0):
    """Paired comparison of pooled recall (all heads) and far recall (top half of frame)
    between baseline A and candidate B, scored on the SAME test frames.
    Returns {metric: dict(delta_pp, lo, hi, p_signflip)}.
      - interval: cluster bootstrap over frames; ONE index matrix shared by both metrics
      - p-value : exact sign-flip test over all 2^F frame-sign patterns (F=10 -> 1024)
    Neither accounts for uncertainty in the cutoff (it was tuned on other frames)."""
    assert A.index.equals(B.index) and A.index.is_unique, "frames must match and be unique"
    F = len(A)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, F, (n_boot, F))                              # shared resamples
    signs = np.array(list(itertools.product([-1.0, 1.0], repeat=F)))   # all sign patterns
    out = {}
    for label, tp, den in (("recall", "tp", "n_true"), ("recall_far", "tp_far", "n_far")):
        assert (A[den] == B[den]).all(), f"{den} differs between settings"
        d = A[den].to_numpy(float)
        diff = B[tp].to_numpy(float) - A[tp].to_numpy(float)           # extra heads found per frame
        point = 100 * diff.sum() / d.sum()                             # pooled delta in pp
        den_b = d[idx].sum(1)
        assert (den_b > 0).all(), "zero denominator in a resample"
        boot = 100 * diff[idx].sum(1) / den_b                          # bootstrap deltas
        lo, hi = np.percentile(boot, [2.5, 97.5])
        null = 100 * (signs @ diff) / d.sum()                          # sign-flip null distribution
        p = float((np.abs(null) >= abs(point) - 1e-9).mean())
        out[label] = dict(delta_pp=point, lo=lo, hi=hi, p_signflip=p)
    return out

def frame_table(det, fids, thr, rows_fn):
    """Per-frame scoring table (index = frame_id) of one detection set at one cutoff."""
    return rows_fn(det, fids, thr).set_index("frame_id").loc[fids]

def matched_precision_cutoff(det, tune_ids, thresholds, target_prec, rows_fn):
    """Cutoff whose pooled precision on the TUNING frames is closest to target_prec,
    so a recall gain cannot come from simply accepting more false boxes."""
    best, gap = None, 9
    for t in thresholds:
        r = rows_fn(det, tune_ids, t)
        prec = r["tp"].sum() / max(r["n_det"].sum(), 1)
        if abs(prec - target_prec) < gap:
            best, gap = t, abs(prec - target_prec)
    return best

def holm(pvals):
    """Holm step-down adjusted p-values for a dict {name: p}."""
    names = sorted(pvals, key=pvals.get); m = len(names); adj = {}; run = 0
    for i, k in enumerate(names):
        run = max(run, min(1.0, (m - i) * pvals[k])); adj[k] = run
    return adj

def run_comparisons(dets, models, configs, base, primary, table, rows_fn,
                    test_ids, tune_ids, thresholds):
    """All (model, setting) vs baseline comparisons, each at (a) its own tuned cutoff and
    (b) a cutoff matched to the baseline's precision. Only `primary` recall is confirmatory
    (Holm-corrected across models); every other row is exploratory."""
    cut = {(r.model, r.setting): r.cutoff for r in table.itertuples()}
    rows = []
    for m in models:
        tA = cut[(m, base)]
        A = frame_table(dets[(m, base)], test_ids, tA, rows_fn)
        rA = rows_fn(dets[(m, base)], tune_ids, tA)
        target = rA["tp"].sum() / max(rA["n_det"].sum(), 1)            # baseline precision (tuning)
        for name in configs:
            if name == base:
                continue
            for mode, tB in (("own_cutoff", cut[(m, name)]),
                             ("matched_precision", matched_precision_cutoff(
                                 dets[(m, name)], tune_ids, thresholds, target, rows_fn))):
                B = frame_table(dets[(m, name)], test_ids, tB, rows_fn)
                for metric, r in paired_delta(A, B).items():
                    rows.append(dict(model=m, setting=name, mode=mode, metric=metric,
                                     cutoff_B=tB, prec_A=A["tp"].sum() / A["n_det"].sum(),
                                     prec_B=B["tp"].sum() / B["n_det"].sum(), **r))
    df = pd.DataFrame(rows)
    df["confirmatory"] = (df.setting == primary) & (df.metric == "recall")
    for mode in df["mode"].unique():                                   # Holm over confirmatory family
        sel = df.index[df.confirmatory & (df["mode"] == mode)]
        for i, a in holm({i: df.loc[i, "p_signflip"] for i in sel}).items():
            df.loc[i, "p_holm"] = a
    df["verdict"] = np.where(df.confirmatory,
                             np.where((df.lo > 0) & (df.p_holm < 0.05), "reliable gain", "not shown"),
                             "exploratory")
    df["meets_5pp_rule"] = df.delta_pp >= 5.0                          # roadmap rule: >= +5 pp recall
    return df

# ---- run it (uses names already defined in Cells 4-6) ----
BASE = "full frame @1920 (E001 baseline)"
PRIMARY = "tiles 640 @1280 (2x)"
assert BASE in CONFIGS and PRIMARY in CONFIGS, "label mismatch with CONFIGS"
res = run_comparisons(dets, MODELS, list(CONFIGS), BASE, PRIMARY, table, rows_for,
                      test_ids, tune_ids, thresholds)
res.to_csv(OUT / "E002_paired_deltas.csv", index=False)               # full precision on disk
show = ["model", "setting", "mode", "metric", "delta_pp", "lo", "hi",
        "p_signflip", "p_holm", "prec_A", "prec_B", "verdict", "meets_5pp_rule"]
print(res[show].round(3).to_string(index=False))

     model                      setting              mode     metric  delta_pp      lo      hi  p_signflip  p_holm  prec_A  prec_B       verdict  meets_5pp_rule
yolov8x.pt tiles 640 @640 (1x, control)        own_cutoff     recall     1.593   0.292   3.005       0.047     NaN    0.68   0.628   exploratory           False
yolov8x.pt tiles 640 @640 (1x, control)        own_cutoff recall_far    -1.006  -3.182   2.073       0.531     NaN    0.68   0.628   exploratory           False
yolov8x.pt tiles 640 @640 (1x, control) matched_precision     recall   -10.825 -12.178  -9.223       0.002     NaN    0.68   0.727   exploratory           False
yolov8x.pt tiles 640 @640 (1x, control) matched_precision recall_far   -13.731 -16.242  -9.982       0.002     NaN    0.68   0.727   exploratory           False
yolov8x.pt         tiles 640 @1280 (2x)        own_cutoff     recall    -3.165  -4.852  -1.149       0.020   0.020    0.68   0.675     not shown           False
yolov8x.pt         tiles 640 @1280

In [9]:
# ===== Cell 7b: equal-weight per-frame robustness check (run after Cell 7) =====
def per_frame_check(dets, models, base, cand, table, rows_fn, test_ids):
    """Robustness check for the confirmatory comparison (cand vs base, each at its own cutoff).
    Treats every test frame EQUALLY (per-frame recall delta), counts frames where cand wins,
    runs an exact sign-flip test on those per-frame deltas, and reports the TEST-set precision
    gap so a threshold that did not transfer from the tuning frames is visible."""
    cut = {(r.model, r.setting): r.cutoff for r in table.itertuples()}
    signs = np.array(list(itertools.product([-1.0, 1.0], repeat=len(test_ids))))
    rows = []
    for m in models:
        A = frame_table(dets[(m, base)], test_ids, cut[(m, base)], rows_fn)
        B = frame_table(dets[(m, cand)], test_ids, cut[(m, cand)], rows_fn)
        assert A.index.equals(B.index)
        dlt = 100 * (B["tp"] / B["n_true"] - A["tp"] / A["n_true"]).to_numpy()   # pp per frame
        null = signs @ dlt / len(dlt)
        pA = A["tp"].sum() / A["n_det"].sum(); pB = B["tp"].sum() / B["n_det"].sum()
        rows.append(dict(model=m, mean_frame_delta_pp=dlt.mean(), min_frame=dlt.min(),
                         max_frame=dlt.max(), frames_cand_better=int((dlt > 0).sum()),
                         frames_worse=int((dlt < 0).sum()), n_frames=len(dlt),
                         p_signflip_equal_weight=float((np.abs(null) >= abs(dlt.mean()) - 1e-9).mean()),
                         test_prec_base=pA, test_prec_cand=pB, prec_gap=pB - pA,
                         precision_transferred=abs(pB - pA) <= 0.03,
                         top_frame_share_of_heads=A["n_true"].max() / A["n_true"].sum()))
    return pd.DataFrame(rows)

chk = per_frame_check(dets, MODELS, BASE, PRIMARY, table, rows_for, test_ids)
chk.to_csv(OUT / "E002_per_frame_check.csv", index=False)
print(chk.round(3).T.to_string())

                                   0           1
model                     yolov8x.pt  yolo26x.pt
mean_frame_delta_pp            -3.06       7.876
min_frame                     -6.208       5.109
max_frame                      3.242      10.656
frames_cand_better                 3          10
frames_worse                       7           0
n_frames                          10          10
p_signflip_equal_weight        0.023       0.002
test_prec_base                  0.68        0.76
test_prec_cand                 0.675       0.688
prec_gap                      -0.006      -0.072
precision_transferred           True       False
top_frame_share_of_heads       0.117       0.117
